# The NSM search integration: RNS Update's load-bearing API

**Source module:** `lib/fetchReports.js` &nbsp;·&nbsp; **Companion to:** the
project README's *"API integration notes"* &nbsp;·&nbsp; **Requires:**
Python 3.9+, `requests`

This notebook carves out and explains `lib/fetchReports.js` — the one piece of
external integration the whole [RNS Update](https://github.com/tomhyde10/rns_update)
app depends on. Every feature (the report list, the RSS/`.ics` feeds, email
digests, push notifications) ultimately calls this module, and this module
calls exactly one thing: an **undocumented, reverse-engineered endpoint** on
`data.fca.org.uk`, the FCA's National Storage Mechanism (NSM).

"Load-bearing" here means two things at once:

- **Everything rests on it.** There is no other data source in the app.
- **It could move under you with no warning.** There is no public API
  reference, no changelog, no key, and no SLA — the request shape below was
  reverse-engineered by watching real browser network traffic, not from any
  spec. The source file's own comments are explicit about this, and this
  notebook keeps that framing rather than presenting the shape as a
  documented contract.

Each section below ports the relevant piece of `lib/fetchReports.js` to
Python — line-for-line where it matters — and runs it, so the *behaviour*
(not just the code) is visible. Where a real network call would be needed,
the cell tries one and falls back to a small schema-accurate stand-in if
there's no connectivity from wherever you're running this — that fallback is
clearly labelled and is **not** a literal captured response.


## Contents

1. [How to use this notebook](#how-to-use-this-notebook)
2. [The request](#1-the-request)
3. [One request per watched company](#2-one-request-per-watched-company)
4. [Calling it for real (with an offline fallback)](#3-calling-it-for-real-with-an-offline-fallback)
5. [Normalising a raw hit into a display-ready report](#4-normalising-a-raw-hit-into-a-display-ready-report)
6. [Filtering: window, categories, and keyword-OR-category](#5-filtering-window-categories-and-keyword-or-category)
7. [The cache: full fetch, cache hit, or delta fetch](#6-the-cache-full-fetch-cache-hit-or-delta-fetch)
8. [Failure handling, and what "load-bearing but fragile" means here](#7-failure-handling-and-what-load-bearing-but-fragile-means-here)
9. [Running the manual end to end](#8-running-the-manual-end-to-end)
10. [Appendix: where this sits in the app](#appendix-where-this-sits-in-the-app)


---

## How to use this notebook

- **Run cells top to bottom.** Later sections reuse functions and variables
  defined earlier (e.g. section 4's `normalise()` is called on section 3's
  `raw_items`), the same as any notebook that builds up a single pipeline.
- **Requirements**: Python 3.9+ and the `requests` package. Nothing else —
  the module being explained (`lib/fetchReports.js`) has zero dependencies
  of its own beyond Node's built-in `fetch`.
- **Network is optional.** Section 3 tries one live call and falls back to a
  labelled offline sample if it can't reach `data.fca.org.uk` — every cell
  after it runs identically either way.
- **Nothing here writes to the real app.** This notebook only reads/derives
  data; it doesn't touch `config/watchlist.js`, the Postgres cache, or any
  running deployment.
- **`print_table()`**, defined in the next cell, is the one helper this
  notebook adds on top of the ported code — a small aligned-table printer
  used for readability, not part of `lib/fetchReports.js` itself.

### The instruction manual: how a report reaches the app, step by step

This is the same sequence `fetchReports()` runs in `lib/fetchReports.js`,
one watched company at a time. Each step names the section below that ports
and runs it.

| # | Step | What happens | Section |
|---|------|--------------|---------|
| 1 | **Build the request** | Assemble the one confirmed-working JSON body for a given LEI, window and page size. | §1 |
| 2 | **Size the request** | Turn the requested day-window into a `size` (items to request), and attach browser-like headers. | §2 |
| 3 | **Check the cache** | Look up this LEI: fresh → skip straight to step 6; stale-but-present → step 4 asks for less; cold → step 4 asks for everything. | §6 |
| 4 | **Call NSM** | POST the request; one call per company, never a multi-company batch. | §3 |
| 5 | **Merge into the cache** | A delta fetch's results are merged with what was already cached, fresh copy winning on collision. | §6 |
| 6 | **Normalise** | Map NSM's raw field names (`headline`, `type`, `publication_date`, ...) onto the app's report shape. | §4 |
| 7 | **Filter** | Keep items inside the requested day-window, then keep ones matching a report category **or** a keyword. | §5 |
| 8 | **Handle failure** | One company failing doesn't fail the batch; only *every* company failing does. | §7 |

Section 8, at the end, chains all of the above into a single function and
runs it — the manual, executed rather than only described.


In [ ]:
def print_table(headers: list, rows: list, max_width: int = 48) -> None:
    """A small dependency-free table printer used throughout this notebook,
    so tabular output stays aligned without pulling in pandas/tabulate for
    something this notebook's own dependency list (Python + requests) can do
    on its own.
    """

    def clip(value) -> str:
        text = str(value)
        return text if len(text) <= max_width else text[: max_width - 1] + "…"

    clipped_rows = [[clip(cell) for cell in row] for row in rows]
    widths = [len(str(h)) for h in headers]
    for row in clipped_rows:
        widths = [max(w, len(cell)) for w, cell in zip(widths, row)]

    def fmt_row(cells) -> str:
        return " | ".join(str(c).ljust(w) for c, w in zip(cells, widths))

    print(fmt_row(headers))
    print("-+-".join("-" * w for w in widths))
    for row in clipped_rows:
        print(fmt_row(row))


---

## 1. The request

```
POST https://api.data.fca.org.uk/search?index=nsm-search
```

No API key. The body below is the *one* shape confirmed to work, reproduced
here exactly rather than "cleaned up" — three details matter enough to call
out:

- **`company_lei`'s value is `["", "<LEI>", "disclose_org", "related_org"]`.**
  The leading empty string and the two flag strings were captured verbatim
  from a real request. Nothing documents what they mean; they're kept as-is
  because deviating from a known-working shape, against an undocumented API,
  is how you find out the hard way that something mattered.
- **`dateCriteria`'s `from` is always `null`.** The window (e.g. "last 7
  days") is *not* sent to NSM at all — it's enforced afterwards, client-side,
  against the dates already present on each returned item. Only `to` is set,
  so the request always says "everything up to now" and the app does the
  actual filtering.
- **Timestamps have no milliseconds.** `Date.prototype.toISOString()`
  produces `...123Z`; a live request with that shape got a 404 ("Unable to
  search the data"). Stripping to `...Z` is the one thing that made it work.


In [ ]:
import json
from datetime import datetime, timezone

NSM_SEARCH_URL = "https://api.data.fca.org.uk/search?index=nsm-search"
NSM_ARTEFACT_BASE = "https://data.fca.org.uk/artefacts/"


def to_iso_no_millis(dt: datetime) -> str:
    """Port of toIsoNoMillis() in lib/fetchReports.js.

    A millisecond-bearing `to` timestamp returned a 404 in a real capture,
    so this strips them to match the one shape known to work.
    """
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def build_request_body(lei: str, to_iso: str, size: int) -> dict:
    """Port of buildRequestBody() in lib/fetchReports.js.

    Mirrors a real captured browser request as closely as possible,
    including `from: null` on both dateCriteria entries and the exact
    company_lei value array shape.
    """
    return {
        "from": 0,
        "size": size,
        "sort": "submitted_date",
        "sortorder": "desc",
        "criteriaObj": {
            "criteria": [
                {"name": "company_lei", "value": ["", lei, "disclose_org", "related_org"]},
                {"name": "latest_flag", "value": "Y"},
            ],
            "dateCriteria": [
                {"name": "publication_date", "value": {"from": None, "to": to_iso}},
                {"name": "submitted_date", "value": {"from": None, "to": to_iso}},
            ],
        },
    }


# Fidelity European Trust plc — one of two LEIs in config/watchlist.js that
# are independently confirmed correct.
FIDELITY_EUROPEAN_TRUST_LEI = "549300UC0QPP7Y0W8056"

to_iso = to_iso_no_millis(datetime.now(timezone.utc))
example_body = build_request_body(FIDELITY_EUROPEAN_TRUST_LEI, to_iso, 100)
print(json.dumps(example_body, indent=2))


---

## 2. One request per watched company

NSM's `company_lei` filter genuinely filters server-side — confirmed by a
real response where every hit matched the requested LEI. That's why the app
makes **one request per watched company** instead of paginating a whole-market
feed and filtering client-side.

`size` scales with the requested time window: a real capture showed roughly
1–2 filings/day for one company, so 15/day gives headroom, floored at 100 and
capped at 1000 so a long window can't request an enormous page.

The request also carries browser-like headers (`Origin`, `Referer`,
`User-Agent`). CORS itself is a browser-only concern and doesn't apply to a
server-to-server call — these are sent in case the endpoint *also* enforces
them server-side as informal bot filtering. That's unconfirmed either way;
it's a defensive guess, not a documented requirement.


In [ ]:
RESULTS_PER_DAY_ESTIMATE = 15
MAX_RESULTS_PER_COMPANY = 1000


def results_per_company(window_days: int) -> int:
    """Port of resultsPerCompany() in lib/fetchReports.js."""
    return min(MAX_RESULTS_PER_COMPANY, max(100, window_days * RESULTS_PER_DAY_ESTIMATE))


BROWSER_LIKE_HEADERS = {
    "content-type": "application/json",
    "accept": "application/json, text/plain, */*",
    "origin": "https://data.fca.org.uk",
    "referer": "https://data.fca.org.uk/",
    "user-agent": "Mozilla/5.0 (compatible; rns-update-notebook/1.0)",
}

print_table(
    ["Window (days)", "size"],
    [[days, results_per_company(days)] for days in (1, 7, 30, 90, 365)],
)


---

## 3. Calling it for real (with an offline fallback)

The cell below tries a live call. If you're running this notebook somewhere
without a route to `data.fca.org.uk` (as in the sandbox this notebook was
written in), it falls back to a **schema-accurate stand-in** — field names
and shapes match the real response format described in `lib/fetchReports.js`
and the project README, but the values are illustrative, not a literal
capture. That fallback is what the rest of this notebook runs against, so
every later cell works identically whether or not you have network access.


In [ ]:
import requests

SAMPLE_RESPONSE_NOTE = (
    "OFFLINE FALLBACK: schema-accurate stand-in, not a real NSM response."
)

# Shaped like a real hit: a Half-year Financial Report, the report type
# confirmed exact for this integration (see REPORT_TYPES below).
sample_hits = [
    {
        "_source": {
            "lei": FIDELITY_EUROPEAN_TRUST_LEI,
            "company": "Fidelity European Trust plc",
            "headline": "Half-yearly Report",
            "type": "Half-year Financial Report",
            "type_code": "IR",
            "publication_date": "2026-09-08T07:00:00Z",
            "submitted_date": "2026-09-08T07:00:00Z",
            "document_date": "2026-09-08",
            "download_link": "sample/fidelity-european-trust-hy-2026.pdf",
            "disclosure_id": "sample-disclosure-1",
        }
    },
    {
        "_source": {
            "lei": FIDELITY_EUROPEAN_TRUST_LEI,
            "company": "Fidelity European Trust plc",
            "headline": "Net Asset Value(s)",
            "type": "Net Asset Value(s)",
            "type_code": "NAV",
            "publication_date": "2026-09-10T07:00:00Z",
            "submitted_date": "2026-09-10T07:00:00Z",
            "document_date": "2026-09-10",
            "html_link": "https://data.fca.org.uk/#/nsm/nsdetails/sample-nav-1",
            "disclosure_id": "sample-disclosure-2",
        }
    },
]

try:
    response = requests.post(
        NSM_SEARCH_URL,
        headers=BROWSER_LIKE_HEADERS,
        json=example_body,
        timeout=8,
    )
    response.raise_for_status()
    data = response.json()
    hits = data.get("hits", {}).get("hits", [])
    print(f"Live call succeeded: {len(hits)} hit(s).")
except Exception as err:
    print(f"Live call failed ({err!r}) — using the offline fallback instead.")
    print(SAMPLE_RESPONSE_NOTE)
    hits = sample_hits

raw_items = [h["_source"] for h in hits if h.get("_source")]
raw_items


---

## 4. Normalising a raw hit into a display-ready report

`normalise()` turns NSM's raw field names into the shape the rest of the app
uses. Three fallback chains matter:

- **`company`**: prefers the name from the local watchlist (`config/watchlist.js`)
  over whatever NSM sends back, so a company you've labelled keeps that label.
- **`title`**: `headline` → `type` → `'(untitled)'`.
- **`publishedAt`**: `publication_date` → `document_date` → `submitted_date`.
- **`url`**: `download_link` is a path *relative* to
  `https://data.fca.org.uk/artefacts/` — confirmed by comparing a CSV
  export's full download URL against the API's relative one for the same
  document. When there's no `download_link`, it falls back to `html_link`.


In [ ]:
# A tiny stand-in for config/watchlist.js, just enough to show the
# company-name fallback in action.
watchlist = [
    {"lei": FIDELITY_EUROPEAN_TRUST_LEI, "name": "Fidelity European Trust plc"},
]


def document_url_of(item: dict):
    """Port of documentUrlOf() in lib/fetchReports.js."""
    if item.get("download_link"):
        return f"{NSM_ARTEFACT_BASE}{item['download_link']}"
    return item.get("html_link")


def normalise(item: dict) -> dict:
    """Port of normalise() in lib/fetchReports.js."""
    watched = next((c for c in watchlist if c["lei"] == item.get("lei")), None)
    return {
        "lei": item.get("lei"),
        "company": (watched and watched["name"]) or item.get("company") or item.get("lei"),
        "title": item.get("headline") or item.get("type") or "(untitled)",
        "category": item.get("type"),
        "publishedAt": item.get("publication_date") or item.get("document_date") or item.get("submitted_date"),
        "url": document_url_of(item),
        "id": item.get("disclosure_id") or item.get("_id"),
        "raw": item,
    }


normalised = [normalise(item) for item in raw_items]
print_table(
    ["Category", "Company", "Title", "Published"],
    [[r["category"], r["company"], r["title"], r["publishedAt"]] for r in normalised],
)
print()
for r in normalised:
    print(f"  {r['title']!r} -> {r['url']}")


---

## 5. Filtering: window, categories, and keyword-OR-category

Because the API request never carries a real lower date bound, **every**
filter that makes a "last 7 days, Half-year + Annual reports only" view
possible happens after the fact, in Python/JS, against the raw item list:

- **Window**: keep items whose date (`publication_date` → `document_date` →
  `submitted_date`) is on or after `now - windowDays`.
- **Category match**: exact, case-insensitive match against `type` — not a
  substring match. `"Half-year"` alone will not match
  `"Half-year Financial Report"`.
- **Keyword match**: a substring match against `headline` OR `type`, **OR'd
  with** the category match (not AND'd) — so a keyword like `"delisting"`
  surfaces a filing even under a report type you haven't ticked. That's a
  deliberate design choice, not an oversight.


In [ ]:
from datetime import timedelta

REPORT_TYPES = {"Half-year Financial Report", "Annual Financial Report"}


def date_cutoff(window_days: int, now: datetime) -> datetime:
    return now - timedelta(days=window_days)


def item_date(item: dict):
    raw = item.get("publication_date") or item.get("document_date") or item.get("submitted_date")
    return datetime.fromisoformat(raw.replace("Z", "+00:00")) if raw else None


def matches_category(item: dict, category_set: set) -> bool:
    t = item.get("type")
    return isinstance(t, str) and t.lower() in category_set


def matches_keyword(item: dict, keyword_norm: str) -> bool:
    if not keyword_norm:
        return False
    headline = item.get("headline")
    t = item.get("type")
    return (isinstance(headline, str) and keyword_norm in headline.lower()) or (
        isinstance(t, str) and keyword_norm in t.lower()
    )


# A deliberately mixed set: one in-category, one keyword-only (wrong
# category), one that matches neither, one outside the window.
demo_now = datetime(2026, 9, 12, tzinfo=timezone.utc)
demo_items = [
    {"headline": "Half-yearly Report", "type": "Half-year Financial Report", "publication_date": "2026-09-08T07:00:00Z"},
    {"headline": "Proposed delisting from the Official List", "type": "Net Asset Value(s)", "publication_date": "2026-09-09T07:00:00Z"},
    {"headline": "Director/PDMR Shareholding", "type": "Director/PDMR Shareholding", "publication_date": "2026-09-10T07:00:00Z"},
    {"headline": "Half-yearly Report", "type": "Half-year Financial Report", "publication_date": "2026-01-01T07:00:00Z"},
]

category_set = {c.lower() for c in REPORT_TYPES}
keyword_norm = "delisting"
cutoff = date_cutoff(7, demo_now)

within_window = [i for i in demo_items if (d := item_date(i)) and d >= cutoff]
matched = [i for i in within_window if matches_category(i, category_set) or matches_keyword(i, keyword_norm)]

print(f"{len(demo_items)} raw -> {len(within_window)} within the 7-day window -> {len(matched)} matched\n")


def matched_via(item: dict) -> str:
    why = []
    if matches_category(item, category_set):
        why.append("category")
    if matches_keyword(item, keyword_norm):
        why.append("keyword")
    return ", ".join(why)


print_table(
    ["Headline", "Type", "Matched via"],
    [[i["headline"], i["type"], matched_via(i)] for i in matched],
)


---

## 6. The cache: full fetch, cache hit, or delta fetch

Hitting an undocumented endpoint once per company on every page load/refresh
is exactly the kind of load a fragile integration shouldn't take. The cache
keys **only on LEI**, not on window/category — because the window and
category filters above are applied client-side against the same raw item
list, a cache entry fetched with a large enough `size` already covers any
smaller, later request for that LEI.

Three outcomes when a company's data is needed:

1. **Cold / not deep enough** → a full fetch for the requested `size`.
2. **Fresh** (within `NSM_CACHE_TTL_MINUTES`, default 10, and deep enough)
   → served straight from the cache, **zero** NSM requests.
3. **Stale but present and deep enough** → a small **delta fetch**: only ask
   NSM for "what's new since last time" (`catchUpSize`), then merge with what
   was already cached. The fresh copy wins on a collision (an amended
   filing), and anything only in the old page is kept as-is. If the delta
   fetch itself fails, the stale cache is served anyway rather than failing
   the company outright — staleness beats an outage.

This mirrors `test/fetchReports.test.js` exactly; the simulation below
reproduces that test's scenario without touching a real cache or network.


In [ ]:
def item_key(item: dict) -> str:
    """Port of itemKey() in lib/fetchReports.js."""
    if item.get("disclosure_id"):
        return item["disclosure_id"]
    if item.get("_id"):
        return item["_id"]
    date = item.get("publication_date") or item.get("document_date") or item.get("submitted_date")
    return f"{item.get('lei')}|{item.get('type')}|{date}|{item.get('headline')}"


def merge_items(old_items: list, fresh_items: list) -> list:
    """Port of mergeItems() in lib/fetchReports.js — fresh wins on collision,
    sorted newest-first."""
    by_key = {}
    for item in old_items:
        by_key[item_key(item)] = item
    for item in fresh_items:
        by_key[item_key(item)] = item

    def date_of(item):
        raw = item.get("submitted_date") or item.get("publication_date") or item.get("document_date")
        return datetime.fromisoformat(raw.replace("Z", "+00:00")) if raw else datetime.min.replace(tzinfo=timezone.utc)

    return sorted(by_key.values(), key=date_of, reverse=True)


def catch_up_size(elapsed_seconds: float) -> int:
    """Port of catchUpSize() in lib/fetchReports.js."""
    elapsed_days = max(1, -(-elapsed_seconds // (24 * 60 * 60)))  # ceil division
    return results_per_company(int(elapsed_days))


# --- Simulation: cold fetch, then a fresh hit, then a stale delta fetch ---
LEI = "AAAAAAAAAAAAAAAAAAAA"
cache: dict = {}
TTL_SECONDS = 60


def is_fresh(cached, size, now_offset_seconds):
    if not cached:
        return False
    age = now_offset_seconds - cached["fetched_at"]
    return age < TTL_SECONDS and cached["size"] >= size


def simulated_nsm_call(items, now_offset_seconds, label):
    print(f"  [t={now_offset_seconds:>4}s] NSM call ({label}) -> {len(items)} item(s)")
    return items


old_report = {"disclosure_id": "id-1", "headline": "Old report", "type": "Half-year Financial Report",
              "publication_date": "2026-09-01T00:00:00Z", "submitted_date": "2026-09-01T00:00:00Z"}
new_report = {"disclosure_id": "id-2", "headline": "New report", "type": "Half-year Financial Report",
              "publication_date": "2026-09-12T00:00:00Z", "submitted_date": "2026-09-12T00:00:00Z"}

size = results_per_company(30)  # 450

print("1) Cold cache -> full fetch")
fetched = simulated_nsm_call([old_report], 0, "full fetch")
cache[LEI] = {"items": fetched, "size": size, "fetched_at": 0}

print("\n2) Same request again, still within the TTL -> served from cache, no NSM call")
cached = cache[LEI]
assert is_fresh(cached, size, 30), "expected a fresh hit"
print(f"  served {len(cached['items'])} item(s) from cache")

print("\n3) TTL has lapsed (t=61s) -> delta fetch for only what's new, merged with the old page")
cached = cache[LEI]
assert not is_fresh(cached, size, 61)
delta_size = catch_up_size(61 - cached["fetched_at"])
delta = simulated_nsm_call([new_report], 61, f"delta fetch, size={delta_size}")
merged = merge_items(cached["items"], delta)
cache[LEI] = {"items": merged, "size": max(cached["size"], size), "fetched_at": 61}
print(f"  merged cache now holds: {[i['headline'] for i in merged]}")

print("\n4) TTL lapses again, but this delta fetch fails -> stale cache is still served")
cached = cache[LEI]
try:
    raise RuntimeError("NSM unavailable (simulated 503)")
except RuntimeError as err:
    print(f"  delta fetch failed ({err}) -> falling back to stale cache")
    print(f"  served: {[i['headline'] for i in cached['items']]}")


---

## 7. Failure handling, and what "load-bearing but fragile" means here

`fetchReports()` fetches every watched company **in parallel**
(`Promise.all`). A single company's request failing doesn't take down the
others — it only returns a hard `502` when **every** company's request has
failed, since an all-failed result is the one case that shouldn't be
silently reported as "0 reports found".

What makes this integration worth treating carefully, summarised from the
source comments and the project README's "API integration notes" /
"Known limitations":

- **No public docs, no key, no SLA.** The request shape was captured from
  real browser traffic, not a spec — it could change or start blocking
  non-browser traffic without any changelog to warn you.
- **`company_lei`'s batching is untested.** Only ever sent/confirmed with one
  LEI at a time; whether NSM's search accepts several LEIs in one request
  (which would cut a 21-company refresh from 21 requests to 1–3) is
  deliberately left unconfirmed rather than assumed.
- **A handful of filings use a different shape** (e.g. a "Direct Upload" PDF
  factsheet with `ContentVersionId`/`html_link` instead of the usual
  RNS/PRN fields) — `normalise()` only relies on fields both shapes share,
  rather than assuming one canonical response schema.
- **No pagination.** `size` scales with the window and is capped, so an
  unusually high-filing-volume company in a long window could silently miss
  older items rather than paginating for them.

None of this is a defect to "fix" so much as the actual shape of depending
on an endpoint nobody promised you. The caching and delta-fetch strategy in
section 6 exists specifically to minimise how often the app leans on that
promise-free endpoint at all.


---

## 8. Running the manual end to end

Everything above is one company's data moving through the eight-step manual
at the top of this notebook. This section chains steps 1, 2, 6 and 7 into a
single function and runs it against the same data section 3 already
fetched (live or offline-fallback) for Fidelity European Trust — steps 3–5
(the cache) already got their own live simulation in section 6, and step 8
(all-companies-failed) is the one case that can't be shown for a single
company that already succeeded, so it's called out rather than faked.


In [ ]:
def explain_fetch_reports_for(lei: str, window_days: int, category_names: set, keyword: str, items: list) -> list:
    """Runs steps 1, 2, 6 and 7 of the instruction manual end to end for one
    company, against the NSM data already fetched for it in section 3.

    Steps 3-5 (cache lookup / NSM call / merge) have their own live
    simulation in section 6, driven by cache timing rather than by a
    company's real data, so they aren't repeated here.
    """
    # A reference "now" derived from the data itself (latest item + 1 day)
    # rather than the wall clock, so this step runs identically whether
    # `items` came from a live call made just now or the fixed-date offline
    # fallback from section 3.
    reference_now = max(item_date(i) for i in items) + timedelta(days=1)

    print("Step 1: build the request")
    to_iso = to_iso_no_millis(reference_now)
    size = results_per_company(window_days)
    build_request_body(lei, to_iso, size)  # same call as section 1 - body not reprinted
    print(f"  POST body built for {lei}, window={window_days}d -> size={size}")

    print("\nStep 2: browser-like headers attached (see BROWSER_LIKE_HEADERS in section 2)")

    print("\nSteps 3-5: cache lookup / NSM call / merge -> see section 6's live simulation")

    print(f"\nStep 6: normalise {len(items)} raw item(s)")
    normalised_items = [normalise(item) for item in items]
    print_table(
        ["Category", "Title", "Published"],
        [[r["category"], r["title"], r["publishedAt"]] for r in normalised_items],
    )

    print(f"\nStep 7: filter to the last {window_days}d, category in {sorted(category_names)} or keyword {keyword!r}")
    cutoff = date_cutoff(window_days, reference_now)
    category_set_local = {c.lower() for c in category_names}
    keyword_norm_local = (keyword or "").lower()
    kept = [
        r
        for item, r in zip(items, normalised_items)
        if (d := item_date(item)) and d >= cutoff
        and (matches_category(item, category_set_local) or matches_keyword(item, keyword_norm_local))
    ]
    print(f"  kept {len(kept)} of {len(normalised_items)}")

    print("\nStep 8: failure handling - not exercised here (this company's fetch already")
    print("         succeeded above); see section 7 for the all-companies-failed case")

    return kept


final_reports = explain_fetch_reports_for(
    FIDELITY_EUROPEAN_TRUST_LEI,
    window_days=7,
    category_names=REPORT_TYPES,
    keyword="delisting",
    items=raw_items,
)

print("\nFinal report list a caller of fetchReports() would see:")
print_table(["Title", "URL"], [[r["title"], r["url"]] for r in final_reports], max_width=60)


---

## Appendix: where this sits in the app

`lib/fetchReports.js` exports `fetchReports()` and `findReport()`, called
from:

- `api/reports.js` / `server.js`'s `/api/reports` — the main report list.
- `server.js`'s `/api/due-dates` — wants years of Half-year/Annual history,
  so it passes a wider `maxDays` than the public route allows.
- `lib/sendDigest.js` / `lib/sendPush.js` — automatic email digests and push
  notifications, via the shared `daysSinceAsWindow()`/`isPublishedAfter()`
  helpers this module also exports.
- `lib/sendNotification.js`'s `/api/notify` — calls `findReport()` to
  **re-derive** a report server-side from a client-supplied `id`, rather
  than trusting a client-authored title/URL for the email it sends.

Everything above traces back to the same one call to
`api.data.fca.org.uk/search?index=nsm-search` — which is exactly why this
module, more than any other in the app, is worth understanding before
touching.
